# Tahap 3 - Data Cleaning

**Tujuan:** membuat data siap analisis tanpa mengarang atau membuang data tanpa alasan. Setiap keputusan dicatat ke `outputs/03_log_cleaning.csv`.

Prinsip: **baris viral tidak dihapus**, karena justru itu yang ingin dipelajari.

In [ ]:
# Sel 1: Muat data + siapkan pencatat keputusan
import os, re
import numpy as np
import pandas as pd

BASE = '/content'
PATH = os.path.join(BASE, 'data', 'tiktok_data.csv')
OUT = os.path.join(BASE, 'outputs')
os.makedirs(OUT, exist_ok=True)

df = pd.read_csv(PATH, encoding='utf-8-sig', dtype={'id_sound': 'string'})
log = []

def catat(keputusan, jumlah, alasan, dampak):
    log.append({'keputusan': keputusan, 'jumlah_baris_terdampak': jumlah,
                'alasan': alasan, 'dampak': dampak})

n_awal = len(df)

# 9 baris memiliki SEMUA kolom inti kosong: tidak bisa dipakai untuk pertanyaan apa pun.
inti = ['duration_seconds', 'human_time', 'author_followers', 'id_sound', 'views', 'likes']
kosong = df[inti].isna().all(axis=1)
n_kosong = int(kosong.sum())
df = df.loc[~kosong].copy()
catat('Hapus baris yang semua kolom intinya kosong', n_kosong,
      'Tanpa durasi, waktu, followers, views, dan likes baris ini tidak bisa dianalisis',
      f'Baris tersisa {len(df)} dari {n_awal}')

In [ ]:
# Sel 2: Durasi 0 detik jadi missing (baris tetap dipakai) + parsing waktu
nol = df['duration_seconds'].eq(0)
n_nol = int(nol.sum())
df.loc[nol, 'duration_seconds'] = np.nan
catat('Ubah durasi 0 detik menjadi missing (baris tidak dihapus)', n_nol,
      '0 detik diduga kode untuk "tidak tercatat"; metrik interaksi baris ini tetap valid',
      'Analisis durasi memakai kelompok "tidak diketahui"')

df['posted_at'] = pd.to_datetime(df['human_time'], errors='coerce')
n_gagal = int(df['posted_at'].isna().sum())
catat('Ubah human_time menjadi posted_at (datetime)', int(df['posted_at'].notna().sum()),
      'Diperlukan untuk jam dan hari posting',
      f'Gagal parse: {n_gagal} baris')

hari_id = {0: 'Senin', 1: 'Selasa', 2: 'Rabu', 3: 'Kamis',
           4: 'Jumat', 5: 'Sabtu', 6: 'Minggu'}
df['post_hour'] = df['posted_at'].dt.hour.astype('Int64')
df['post_day'] = df['posted_at'].dt.dayofweek.map(hari_id)

bins = [-np.inf, 15, 30, 60, np.inf]
labels = ['<15 detik', '15-30 detik', '30-60 detik', '≥60 detik']
df['duration_group'] = pd.cut(df['duration_seconds'], bins=bins, right=False, labels=labels)
df['duration_group'] = df['duration_group'].astype('object').fillna('tidak diketahui')
catat('Buat kelompok durasi (<15, 15-30, 30-60, ≥60 detik)', len(df),
      'Memudahkan perbandingan; 60 detik masuk kelompok ≥60',
      'Kelompok "tidak diketahui" untuk durasi 0/missing')

In [ ]:
# Sel 3: Rapikan hashtag, tutup data pribadi di caption, buat like_rate
def bersihkan_hashtags(x):
    if pd.isna(x):
        return '', 0
    tokens = [t.strip().lstrip('#').lower() for t in str(x).split(',')]
    tokens = [t for t in tokens if t]
    unik = list(dict.fromkeys(tokens))
    return ', '.join(unik), len(unik)

res = df['hashtags'].apply(bersihkan_hashtags)
df['hashtags_clean'] = res.map(lambda t: t[0])
df['n_hashtags'] = res.map(lambda t: t[1])
catat('Hashtag: trim, lowercase, hapus #, deduplikasi per baris', len(df),
      'Agar pengelompokan tidak terpecah karena beda huruf/spasi',
      'Kolom hashtags_clean dipakai untuk analisis')
n_ht = int(df['hashtags'].isna().sum())
catat('Hashtag kosong dianggap 0 hashtag', n_ht,
      'Deskripsi Kaggle menyebut kolom hashtag bersifat optional',
      'n_hashtags = 0, bukan missing')

# Tutup akun/kontak: aturan portofolio melarang menyimpan nama akun asli & URL profil.
MENTION = re.compile(r'@[^\s]+', re.UNICODE)
KONTAK = re.compile(r'(?i)\b(instagram|insta|inst|ig|telegram|t\.me|whatsapp|wa|line|snapchat|snap|vk)\s*[:=\-–—]\s*[@\w.\-]+', re.UNICODE)
NOMOR = re.compile(r'(?<!\d)(?:\+?\d[\d\-\s()]{7,}\d)(?!\d)')

def tutup_akun(x):
    if pd.isna(x):
        return ''
    s = str(x)
    s = KONTAK.sub('[AKUN]', s)
    s = MENTION.sub('[AKUN]', s)
    s = NOMOR.sub('[NOMOR]', s)
    return s.strip()

df['caption_masked'] = df['text_part'].apply(tutup_akun)
n_ubah = int((df['text_part'].fillna('') != df['caption_masked']).sum())
n_kosong_cap = int(df['text_part'].isna().sum())
catat('Tutup @akun, kontak, dan nomor di caption', n_ubah,
      'Tidak menyimpan nama akun asli atau URL profil di repo',
      'Teks tetap dipakai untuk analisis kata kunci')
catat('Caption kosong dianggap panjang 0', n_kosong_cap,
      'Kolom caption bersifat optional',
      'caption_length = 0')
df['caption_length'] = df['caption_masked'].str.len()

# comments & shares tidak ada => engagement rate definisi awal tidak bisa dihitung.
df['like_rate'] = df['likes'] / df['views']
catat('Buat like_rate = likes/views sebagai proxy', len(df),
      'Engagement rate butuh comments & shares yang tidak tersedia',
      'Disebut proxy; comment_rate/share_rate tidak dibuat')

In [ ]:
# Sel 4: Cek outlier (TIDAK dihapus) dan susun data final
for c in ['views', 'likes']:
    q1, q3 = df[c].quantile([.25, .75])
    batas = q3 + 1.5 * (q3 - q1)
    n = int((df[c] > batas).sum())
    catat(f'Outlier {c} tidak dihapus (batas IQR {batas:,.0f})', n,
          'Video viral adalah objek penelitian; menghapusnya menghilangkan fenomena utama',
          'EDA pakai median, persentil, log skala, dan perbandingan top 1%')

final = pd.DataFrame({
    'video_id': df['id_video'].astype('int64'),
    'posted_at': df['posted_at'].dt.strftime('%Y-%m-%d %H:%M:%S'),
    'post_hour': df['post_hour'],
    'post_day': df['post_day'],
    'duration_sec': df['duration_seconds'],
    'duration_group': df['duration_group'],
    'caption_masked': df['caption_masked'],
    'caption_length': df['caption_length'],
    'hashtags_clean': df['hashtags_clean'],
    'n_hashtags': df['n_hashtags'],
    'sound_id': df['id_sound'],
    'author_followers': df['author_followers'],
    'author_total_likes': df['author_likes'],
    'views': df['views'],
    'likes': df['likes'],
    'like_rate': df['like_rate'].round(6),
})

# Pemeriksaan akhir: tidak boleh ada identitaspribadi.
terlarang = {'user_name', 'username', 'author', 'author_nickname',
             'avatar', 'url', 'video_url', 'profile'}
assert not (terlarang & set(final.columns))
assert not final['caption_masked'].str.contains('@', regex=False).any()
assert not final['caption_masked'].str.contains(r'https?://|www\.', regex=True).any()
assert final['video_id'].is_unique

final.to_csv(os.path.join(OUT, 'tiktok_clean.csv'), index=False, encoding='utf-8-sig')
pd.DataFrame(log).to_csv(os.path.join(OUT, '03_log_cleaning.csv'), index=False, encoding='utf-8-sig')
print('Data bersih:', final.shape)
print('Keputusan tercatat:', len(log))
print(final.isna().sum().to_string())

## Ringkasan keputusan cleaning

| Keputusan | Jumlah | Alasan singkat |
|---|---:|---|
| Hapus baris semua kolom inti kosong | 9 | Tidak bisa dipakai untuk pertanyaan apa pun |
| Durasi 0 → missing | 16 | Diduga kode missing; baris tetap dipakai untuk views/likes |
| `human_time` → `posted_at` | 751 | Untuk jam & hari posting |
| Kelompok durasi | 751 | <15, 15–30, 30–60, ≥60 detik + "tidak diketahui" |
| Hashtag lowercase/trim/dedupe | 751 | Hindari kategori ganda karena beda huruf |
| Hashtag kosong → 0 | 55 | Kolom bersifat optional |
| Tutup akun/kontak di caption | 65 | Privasi: tanpa nama akun asli/URL profil |
| Caption kosong → panjang 0 | 93 | Kolom bersifat optional |
| `like_rate` proxy | 751 | Tidak ada comments & shares |
| Outlier views/likes tidak dihapus | 132 / 121 | Viral justru menjadi fokus analisis |

Detail lengkap ada di `outputs/03_log_cleaning.csv`.

## Hasil eksekusi

- Data bersih: **751 baris × 16 kolom**, tersimpan di `outputs/tiktok_clean.csv`.
- Kelompok durasi: ≥60 detik 345, <15 detik 164, 15–30 detik 133, 30–60 detik 93, tidak diketahui 16.
- Hari posting: Rabu 236, Kamis 111, Senin 107, Jumat 99, Minggu 68, Selasa 68, Sabtu 62.
- `like_rate`: median 0,0484 (4,84%); rentang 0,0022–0,2864.
- Pemeriksaan akhir lolos: tidak ada `@`, URL, atau kolom identitas di data bersih.

**Lanjut:** Tahap 4 - Exploratory Data Analysis.